In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq

In [4]:
load_dotenv()

llm = ChatGroq(model="llama-3.1-8b-instant")

In [5]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [6]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [7]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [8]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [9]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.',
 'explanation': 'The joke is a play on words, using a pun to create a humorous effect. The phrase "feeling a little crusty" typically refers to someone or something that is feeling grumpy or irritable. However, in this joke, it\'s applied to a pizza, which has a crust as a part of its structure.\n\nThe punchline is funny because it takes a common phrase and gives it a literal twist, referencing the pizza\'s crust. It\'s a clever and lighthearted way to make a connection between the pizza\'s physical characteristic and its emotional state, creating a humorous and relatable joke.'}

In [10]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.', 'explanation': 'The joke is a play on words, using a pun to create a humorous effect. The phrase "feeling a little crusty" typically refers to someone or something that is feeling grumpy or irritable. However, in this joke, it\'s applied to a pizza, which has a crust as a part of its structure.\n\nThe punchline is funny because it takes a common phrase and gives it a literal twist, referencing the pizza\'s crust. It\'s a clever and lighthearted way to make a connection between the pizza\'s physical characteristic and its emotional state, creating a humorous and relatable joke.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13cb7f-7ad9-67e0-8002-83b247831e7d'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-20T12:53:45.722637+00:00', parent_config={'configurable': {'thread_id': '1'

In [11]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.', 'explanation': 'The joke is a play on words, using a pun to create a humorous effect. The phrase "feeling a little crusty" typically refers to someone or something that is feeling grumpy or irritable. However, in this joke, it\'s applied to a pizza, which has a crust as a part of its structure.\n\nThe punchline is funny because it takes a common phrase and gives it a literal twist, referencing the pizza\'s crust. It\'s a clever and lighthearted way to make a connection between the pizza\'s physical characteristic and its emotional state, creating a humorous and relatable joke.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13cb7f-7ad9-67e0-8002-83b247831e7d'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-20T12:53:45.722637+00:00', parent_config={'configurable': {'thread_id': '1